# Talk to Your Pipeline — Live Demo
**Skills O'Clock | EMBL Heidelberg**

This notebook runs `nf-core/rnaseq` on 4 samples from Himes et al. 2014 (GSE52778):
dexamethasone-treated vs untreated human airway smooth muscle cells.

**Dataset:** [GEO GSE52778](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE52778) | [Paper (PLoS ONE 2014)](https://doi.org/10.1371/journal.pone.0099625)

**Runtime:** ~45–60 min on Colab free tier (first run builds the STAR index)

---
⚠️ **Before running:** Go to `Runtime > Change runtime type` and select **High-RAM** if available.

## 1. Mount Google Drive
Results are saved to your Drive so they persist after the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORKDIR = '/content/drive/MyDrive/talk-to-your-pipeline'
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print(f'Working directory: {os.getcwd()}')

## 2. Install Java and Nextflow

In [ ]:
%%bash
# Install Java 17
apt-get install -y -q default-jdk
java -version

In [ ]:
%%bash
# Install Nextflow
curl -s https://get.nextflow.io | bash
mv nextflow /usr/local/bin/
nextflow -version

## 3. Install Conda (via condacolab)
nf-core/rnaseq uses the `conda` profile on Colab since Docker is not available.

In [ ]:
# This restarts the runtime — that's expected
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
%%bash
# Install mamba for faster conda environment solving
conda install -y -q -n base -c conda-forge mamba
mamba --version

## 4. Set up the nextflow.config
Caps resource usage to stay within Colab's free tier limits.

In [ ]:
nextflow_config = """
// Use mamba for faster conda environment creation
conda.useMamba = true
"""

with open('nextflow.config', 'w') as f:
    f.write(nextflow_config)

print('nextflow.config written')

## 5. Create the input files
SRA accession IDs and the samplesheet.

In [ ]:
import os
os.makedirs('data', exist_ok=True)

# SRA accession IDs for fetchngs
sample_ids = """SRR1039508
SRR1039509
SRR1039512
SRR1039513
"""
with open('data/sample-ids.csv', 'w') as f:
    f.write(sample_ids)

print('data/sample-ids.csv written')

## 6. Download chr22 reference
Using a chromosome 22-only reference keeps the STAR index small (~3 GB RAM) so it fits within Colab's free tier.
On the full GRCh38 genome the alignment rate would be ~85%; on chr22 it will be ~4% — expected, since reads come from all chromosomes.

In [ ]:
%%bash
mkdir -p data/reference

# chr22 FASTA from Ensembl GRCh38 release 110
curl -s -L "https://ftp.ensembl.org/pub/release-110/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.chromosome.22.fa.gz" \
    -o data/reference/chr22.fa.gz && gunzip -f data/reference/chr22.fa.gz

# Filter full GTF to chr22 entries only
curl -s -L "https://ftp.ensembl.org/pub/release-110/gtf/homo_sapiens/Homo_sapiens.GRCh38.110.gtf.gz" \
    | zcat | awk '$1 == "22"' > data/reference/chr22.gtf

echo "Reference files:"
ls -lh data/reference/

## 7. Create the params file

In [ ]:
params = """
# nf-core/rnaseq — Himes et al. 2014 (GSE52778)
# Human airway smooth muscle cells: dexamethasone vs untreated
# 4 samples (2 donors x 2 conditions), chr22 reference for Colab compatibility

input:  "data/samplesheet.csv"
outdir: "results/himes"

# Reference — chr22 only (fits in Colab free tier RAM)
fasta: "data/reference/chr22.fa"
gtf:   "data/reference/chr22.gtf"

# Alignment
aligner: star_salmon
save_reference: true

# Trimming
trimmer: trimgalore
min_trimmed_reads: 10000

# QC
skip_dupradar: true
skip_deseq2_qc: false

# Resource caps for Colab free tier (2 vCPUs, ~15 GB RAM)
max_cpus:   2
max_memory: "12.GB"
max_time:   "24.h"
"""

with open('params.yml', 'w') as f:
    f.write(params)

print('params.yml written')

## 8. Download data with nf-core/fetchngs
fetchngs downloads the FASTQs from SRA and auto-generates an nf-core samplesheet.

In [ ]:
%%bash
nextflow run nf-core/fetchngs -r dev \
    --input data/sample-ids.csv \
    --outdir data/fetchngs_results \
    --nf_core_pipeline rnaseq \
    -profile conda \
    -c nextflow.config

In [ ]:
# Write the samplesheet pointing to the downloaded FASTQs
samplesheet = """sample,fastq_1,fastq_2,strandedness
N61311_untreated,data/fetchngs_results/fastq/SRR1039508_1.fastq.gz,data/fetchngs_results/fastq/SRR1039508_2.fastq.gz,auto
N61311_dex,data/fetchngs_results/fastq/SRR1039509_1.fastq.gz,data/fetchngs_results/fastq/SRR1039509_2.fastq.gz,auto
N052611_untreated,data/fetchngs_results/fastq/SRR1039512_1.fastq.gz,data/fetchngs_results/fastq/SRR1039512_2.fastq.gz,auto
N052611_dex,data/fetchngs_results/fastq/SRR1039513_1.fastq.gz,data/fetchngs_results/fastq/SRR1039513_2.fastq.gz,auto
"""

with open('data/samplesheet.csv', 'w') as f:
    f.write(samplesheet)

print('data/samplesheet.csv written')

## 9. Run nf-core/rnaseq
⏱ First run takes ~45–60 min (builds STAR index + aligns 4 samples). Use `-resume` to restart from where it left off if the session disconnects.

In [ ]:
%%bash
nextflow run nf-core/rnaseq -r 3.25.0 \
    -profile conda \
    -c nextflow.config \
    -params-file params.yml \
    -resume

## 10. View the MultiQC report
Renders the HTML report inline in the notebook.

In [ ]:
import glob
from IPython.display import IFrame, display

reports = glob.glob('results/himes/multiqc/multiqc_report.html')
if reports:
    display(IFrame(src=reports[0], width='100%', height=800))
else:
    print('MultiQC report not found — check if the pipeline completed successfully.')

## 11. What's next? Ask Copilot

Paste this into GitHub Copilot Chat in VS Code:

```
nf-core/rnaseq has finished. I have Salmon quantification files for 4 samples
in results/himes/star_salmon/. My experiment compares dexamethasone vs untreated
human airway smooth muscle cells with 2 donors per condition.

What should I do next to find differentially expressed genes? What tool
should I use, and write me the minimal R code to get started with DESeq2
using the correct paired-donor design.
```